# TextMamba3D — A100 Training Pipeline (V5.0)

**V5.0: Mamba-3 Complex-Valued SSM Backbone**

| Feature | Description |
|---------|-------------|
| SSM Backend | Mamba-3 complex-valued SSM with RoPE + trapezoidal discretization |
| d_state | 16 (same as V4.5, isolates Mamba-3 effect) |
| headdim | 48 (divides all stage dims: 48, 96, 192, 384) |
| A100 40GB | batch_size=2, gradient_accumulation=2 (effective batch=4) |
| Checkpoint | **Incompatible** with Mamba-1 — must train from scratch |

Config: `configs/textbrats_a100_v5.yaml`

> **Key change from V4.6:** swaps the inner SSM from Mamba-1 to Mamba-3 across
> the entire image encoder/decoder pipeline. The text encoder (lightweight
> MambaLayer adapter over frozen PubMedBERT) stays on Mamba-1.
> Targets TC Dice regression via complex-valued rotational dynamics across
> 3-axis cross-scan (DHW, HWD, WDH).

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# GPU check
!nvidia-smi 2>/dev/null || echo "No GPU detected (CPU mode)"

# Install dependencies (Mamba2 from PyPI, stable)
!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel tensorboard pyyaml tqdm

# Verify
import mamba_ssm
print('mamba_ssm version:', mamba_ssm.__version__)
print('Mamba2 available:', hasattr(mamba_ssm, 'Mamba2'))
assert hasattr(mamba_ssm, 'Mamba2'), 'Mamba2 not found!'
print('OK')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thu Mar 19 12:31:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             51W /  400W |    1137MiB /  40960MiB |      0%      Default |
|          

In [2]:
import os, zipfile, shutil, subprocess, time

REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'

# Code source: git clone (public repo)
# If old zip-extracted dir exists (no .git), remove and re-clone
git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    print(f'Removing old non-git code at {REPO_DIR}...')
    shutil.rmtree(REPO_DIR)

if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
    print(f'Updated existing repo at {REPO_DIR}')
else:
    # Clone with retry
    for attempt in range(1, 4):
        print(f'Cloning (attempt {attempt}/3)...')
        ret = subprocess.run(
            ['git', 'clone', '--depth', '1', 'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True
        )
        if ret.returncode == 0 and os.path.exists(os.path.join(REPO_DIR, 'models/textmamba3d.py')):
            break
        print(f'  Failed (code {ret.returncode}): {ret.stderr.strip()}')
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        if attempt < 3:
            time.sleep(5 * attempt)
    else:
        raise RuntimeError(f'Clone failed after 3 attempts. Last error: {ret.stderr.strip()}')
    os.chdir(REPO_DIR)
    print(f'Cloned to {REPO_DIR}')

print(f'Working directory: {os.getcwd()}')

# Extract BraTS data from Drive
DATA_ZIP = os.path.join(DRIVE_BASE, "TextBraTS_data.zip")
DATA_DIR = os.path.join(REPO_DIR, "data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData")

if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    if os.path.exists(DATA_ZIP):
        print(f"Extracting {DATA_ZIP}...")
        with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
            zf.extractall(os.path.dirname(DATA_DIR))
        if os.path.exists(DATA_DIR):
            print(f"Data extracted. Cases: {len(os.listdir(DATA_DIR))}")
        else:
            print(f"ERROR: Expected path not found after extraction: {DATA_DIR}")
            print("Actual contents:", os.listdir(os.path.dirname(DATA_DIR)))
    else:
        print(f"ERROR: {DATA_ZIP} not found on Drive")
else:
    print(f"Data already exists. Cases: {len(os.listdir(DATA_DIR))}")

# Count samples
if os.path.exists(DATA_DIR):
    cases = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
    print(f"Total BraTS cases: {len(cases)}")

Updated existing repo at /content/TextMamba3D
Working directory: /content/TextMamba3D
Data already exists. Cases: 371
Total BraTS cases: 369


In [3]:
import sys, yaml, inspect
sys.path.insert(0, '.')

# Verify V5.0 modules (using Mamba2 as SSM backend)
from models.mamba_block import (
    _create_ssm, _auto_headdim, MAMBA3_AVAILABLE,
    MambaBlock, BiMambaBlock, CrossScanBiMamba3DBlock,
)
print(f'MAMBA3_AVAILABLE (Mamba2 alias): {MAMBA3_AVAILABLE}')
assert MAMBA3_AVAILABLE, 'Mamba2 not available! Check mamba-ssm installation.'

# Verify _auto_headdim for V5.0 stage dimensions
for dim in [48, 96, 192, 384]:
    d_inner = dim * 2  # expand=2
    hd = _auto_headdim(d_inner)
    assert d_inner % hd == 0, f'headdim={hd} invalid for d_inner={d_inner}'
    print(f'  Stage dim={dim} -> d_inner={d_inner} -> auto headdim={hd}')

# Verify TextMamba3D has V5.0 params
from models.textmamba3d import TextMamba3D
sig = inspect.signature(TextMamba3D.__init__)
for param in ['use_mamba3', 'headdim']:
    assert param in sig.parameters, f'TextMamba3D missing V5.0 param: {param}!'
print('TextMamba3D: use_mamba3, headdim params present')

# Verify config
with open('configs/textbrats_a100_v5.yaml') as f:
    cfg = yaml.safe_load(f)
assert cfg['model']['use_mamba3'] is True, 'use_mamba3 should be True'
assert cfg['model']['headdim'] == 48
assert cfg['data']['batch_size'] == 2
assert cfg['training']['gradient_accumulation'] == 2
print(f'Config verified: use_mamba3=True (Mamba2 backend), headdim=48')
print(f'  batch={cfg["data"]["batch_size"]}, accum={cfg["training"]["gradient_accumulation"]}')

# Mamba2 smoke test on CUDA
import torch
assert torch.cuda.is_available(), 'CUDA GPU required!'
from mamba_ssm import Mamba2
device = torch.device('cuda')
ssm = Mamba2(d_model=48, d_state=16, expand=2, headdim=48).to(device)
x = torch.randn(1, 64, 48).to(device)
out = ssm(x)
assert out.shape == x.shape, f'Shape mismatch: {out.shape} != {x.shape}'
print(f'Mamba2 smoke test: {x.shape} -> {out.shape} OK')
del ssm, x, out
torch.cuda.empty_cache()

print()
print('All V5.0 modules verified!')

MAMBA3_AVAILABLE (Mamba2 alias): True
  Stage dim=48 -> d_inner=96 -> auto headdim=48
  Stage dim=96 -> d_inner=192 -> auto headdim=64
  Stage dim=192 -> d_inner=384 -> auto headdim=64
  Stage dim=384 -> d_inner=768 -> auto headdim=64
TextMamba3D: use_mamba3, headdim params present
Config verified: use_mamba3=True (Mamba2 backend), headdim=48
  batch=2, accum=2
Mamba2 smoke test: torch.Size([1, 64, 48]) -> torch.Size([1, 64, 48]) OK

All V5.0 modules verified!


In [4]:
import os, sys, zipfile
os.chdir(REPO_DIR)

DATA_DIR = "./data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
ET_CACHE_ZIP = os.path.join(DRIVE_BASE, "et_enriched.zip")

# Safety check: verify dataset exists and is non-empty
cases = sorted(
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
) if os.path.isdir(DATA_DIR) else []
if not cases:
    raise RuntimeError(
        f"No BraTS cases found in {DATA_DIR}. "
        "Check that data extraction completed successfully."
    )

sample_case = cases[0]
sample_enriched = os.path.join(DATA_DIR, sample_case, f"{sample_case}_et_enriched.txt")

if os.path.exists(sample_enriched):
    # Already generated (same runtime)
    count = sum(
        1 for d in cases
        if os.path.exists(os.path.join(DATA_DIR, d, f"{d}_et_enriched.txt"))
    )
    print(f"ET-enriched text already present for {count} cases, skipping")
    for case in cases[:3]:
        path = os.path.join(DATA_DIR, case, f"{case}_et_enriched.txt")
        if os.path.exists(path):
            with open(path, 'r') as f:
                print(f"  {case}: {f.read().strip()[:120]}...")

elif os.path.exists(ET_CACHE_ZIP):
    # Restore from Drive cache
    print(f"Restoring ET-enriched text from {ET_CACHE_ZIP}...")
    with zipfile.ZipFile(ET_CACHE_ZIP, 'r') as zf:
        zf.extractall(DATA_DIR)
    count = sum(
        1 for d in cases
        if os.path.exists(os.path.join(DATA_DIR, d, f"{d}_et_enriched.txt"))
    )
    print(f"Restored ET-enriched text for {count} cases from Drive cache")

else:
    # Generate + cache to Drive
    print("Generating ET-enriched text descriptions from T1ce images...")
    sys.path.insert(0, '.')
    from data.et_text_enrichment import process_all_cases
    results = process_all_cases(DATA_DIR)

    no_enhancement = sum(1 for desc in results.values() if "No significant" in desc)
    total = len(results)
    print(f"Total: {total}, No enhancement: {no_enhancement} ({no_enhancement/total*100:.1f}%)")

    # Cache to Drive for next runtime
    with zipfile.ZipFile(ET_CACHE_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
        for case_dir in cases:
            et_file = os.path.join(DATA_DIR, case_dir, f"{case_dir}_et_enriched.txt")
            if os.path.exists(et_file):
                zf.write(et_file, os.path.join(case_dir, f"{case_dir}_et_enriched.txt"))
    print(f"Cached ET text to {ET_CACHE_ZIP}")

# Hard check: fail fast if ET-enriched text is missing
et_count = sum(
    1 for d in cases
    if os.path.exists(os.path.join(DATA_DIR, d, f'{d}_et_enriched.txt'))
)
if et_count == 0:
    raise RuntimeError(
        f'et_enriched=true in config but 0 ET-enriched text files found in {DATA_DIR}. '
        'Run the ET text generation step first (see V4.5 training notebook Cell 4).'
    )
print(f'ET-enriched text verified: {et_count}/{len(cases)} cases')


ET-enriched text already present for 369 cases, skipping
  BraTS20_Training_001: Solid enhancing lesion on T1ce, in the right anterior inferior region, with irregular margins, showing moderate enhancem...
  BraTS20_Training_002: Solid enhancing lesion on T1ce, in the left anterior inferior region, with irregular margins, showing moderate enhanceme...
  BraTS20_Training_003: Solid enhancing lesion on T1ce, in the left anterior inferior region, with irregular margins, showing focal enhancement,...
ET-enriched text verified: 369/369 cases


In [5]:
# Smoke test: 2 samples, 1 epoch
import os
os.chdir(REPO_DIR)

print('Running smoke test...')
!python -u train.py \
    --config configs/textbrats_a100_v5.yaml \
    --max-samples 2 \
    --max-epochs 1 \
    --no-text-ratio 0.0 \
    --grad-accum 1

import torch
if torch.cuda.is_available():
    peak = torch.cuda.max_memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'Peak GPU memory: {peak:.1f} / {total:.0f} GB')
    torch.cuda.reset_peak_memory_stats()

Running smoke test...
2026-03-19 12:31:55.124734: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773923515.147497   25915 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773923515.155127   25915 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773923515.175127   25915 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773923515.175155   25915 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773923515.175158   25915 computation_placer.cc:177] 

## Training (A100 40GB)

| Parameter | Value | Note |
|-----------|-------|------|
| batch_size | 2 | Reduced from 4 (Mamba3 d_state=64 uses more memory) |
| gradient_accumulation | 2 | Maintains effective batch size of 4 |
| gradient_checkpointing | true | Required for A100 40GB |
| sw_batch_size | 2 | Sliding window inference |
| num_workers | 4 | Colab A100 |
| d_state | 16 | Same as V4.5 (isolate Mamba-3 effect) |
| headdim | 48 | Divides all stage dims |

> **Warning:** Mamba-3 weights are architecturally incompatible with Mamba-1.
> V5.0 must train from scratch — do NOT resume from V4.x checkpoints.

In [6]:
import os, shutil, glob

DRIVE_CKPT = os.path.join(DRIVE_BASE, "checkpoints")
os.makedirs(DRIVE_CKPT, exist_ok=True)

def sync_checkpoints_to_drive():
    local_ckpt = os.path.join(REPO_DIR, "checkpoints")
    if not os.path.exists(local_ckpt):
        return
    for f in glob.glob(os.path.join(local_ckpt, "*.pth")):
        dst = os.path.join(DRIVE_CKPT, os.path.basename(f))
        shutil.copy2(f, dst)
    print(f"Synced checkpoints to {DRIVE_CKPT}")

# Clean local checkpoints (V5.0 trains from scratch)
for f in glob.glob(os.path.join(REPO_DIR, "checkpoints/*.pth")):
    os.remove(f)
print("Cleaned local checkpoints for V5.0 fresh start")
print("(V4.x checkpoints preserved on Drive but incompatible with V5.0)")

Cleaned local checkpoints for V5.0 fresh start
(V4.x checkpoints preserved on Drive but incompatible with V5.0)


In [7]:
import subprocess, shutil
os.chdir(REPO_DIR)
os.environ["DRIVE_CKPT_DIR"] = DRIVE_CKPT

# V5.0 training: Mamba-3 SSM backbone + ET-Enriched (A100 40GB)
# batch=2, grad-accum=2 -> effective batch=4
ret = subprocess.run(
    ['python', '-u', 'train.py',
     '--config', 'configs/textbrats_a100_v5.yaml',
     '--no-text-ratio', '0.15',
     '--grad-accum', '2'],
    cwd=REPO_DIR,
)
if ret.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {ret.returncode}. Check output above.")

# Verify checkpoint was produced before syncing
local_best = os.path.join(REPO_DIR, "checkpoints/best.pth")
local_last = os.path.join(REPO_DIR, "checkpoints/last.pth")
if not os.path.exists(local_best) and not os.path.exists(local_last):
    raise RuntimeError("Training completed but no checkpoint was saved. Check train.py output.")

# Sync and save
sync_checkpoints_to_drive()

best_ckpt = os.path.join(DRIVE_CKPT, "best_v5.0.pth")
if os.path.exists(local_best):
    shutil.copy2(local_best, best_ckpt)
    print(f"Best checkpoint saved: {best_ckpt}")

Synced checkpoints to /content/drive/MyDrive/TextMamba3D/checkpoints
Best checkpoint saved: /content/drive/MyDrive/TextMamba3D/checkpoints/best_v5.0.pth


## Evaluation

In [8]:
os.chdir(REPO_DIR)

ckpt = os.path.join(REPO_DIR, "checkpoints/best.pth")
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, "best_v5.0.pth")

if os.path.exists(ckpt):
    print("=" * 60)
    print("Evaluation: With Text (Mamba3 + SeqCA)")
    print("=" * 60)
    !python evaluate_full.py \
        --config configs/textbrats_a100_v5.yaml \
        --checkpoint "{ckpt}" \
        --split test \
        --use-text \
        --overlap 0.5

    print()

    print("=" * 60)
    print("Evaluation: Without Text (fusion bypassed)")
    print("=" * 60)
    !python evaluate_full.py \
        --config configs/textbrats_a100_v5.yaml \
        --checkpoint "{ckpt}" \
        --split test \
        --no-text \
        --overlap 0.5

    print()
    print("=" * 60)
    print("Compare: with-text Dice - without-text Dice = text guidance delta")
    print("V4.6 baseline: ~88% Mean Dice")
    print("V5.0 target: >= 88% with improved TC class")
    print("Primary metric: TC Dice improvement (complex SSM -> better rotational dynamics)")
    print("=" * 60)
else:
    print(f"No checkpoint found at {ckpt}")
    print("Run training first")

Evaluation: With Text (Mamba3 + SeqCA)
Loading weights: 100% 199/199 [00:00<00:00, 1073.28it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from differ

In [9]:
import matplotlib.pyplot as plt

# Fill in actual results after evaluation
v46_dice = {'ET': 0.0, 'TC': 0.0, 'WT': 0.0, 'Mean': 88.0}  # V4.6 baseline
v50_dice = {'ET': 0.0, 'TC': 0.0, 'WT': 0.0, 'Mean': 0.0}   # Fill after eval

if v50_dice['Mean'] == 0.0:
    print("V5.0 results not yet filled in.")
    print("Update v46_dice and v50_dice dictionaries after evaluation, then re-run this cell.")
    print()
    print("Key metrics to watch:")
    print("  - TC Dice: primary success metric (complex SSM targets rotational dynamics)")
    print("  - Mean Dice: should be >= V4.6 baseline (~88%)")
    print("  - ET Dice: should not regress")
else:
    labels = list(v46_dice.keys())
    v46_vals = list(v46_dice.values())
    v50_vals = list(v50_dice.values())

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart comparison
    x = range(len(labels))
    w = 0.35
    ax1.bar([i - w/2 for i in x], v46_vals, w, label='V4.6 (Mamba-1)', color='steelblue', alpha=0.8)
    ax1.bar([i + w/2 for i in x], v50_vals, w, label='V5.0 (Mamba-3)', color='coral', alpha=0.8)
    ax1.set_ylabel('Dice (%)')
    ax1.set_title('V4.6 vs V5.0 Dice Comparison')
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)

    # Delta chart
    deltas = [v50 - v46 for v46, v50 in zip(v46_vals, v50_vals)]
    colors = ['green' if d >= 0 else 'red' for d in deltas]
    ax2.bar(labels, deltas, color=colors, alpha=0.8)
    ax2.axhline(y=0, color='black', linewidth=0.5)
    ax2.set_ylabel('Delta (%)')
    ax2.set_title('V5.0 - V4.6 Improvement')
    ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('v50_a100_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: v50_a100_comparison.png")

V5.0 results not yet filled in.
Update v46_dice and v50_dice dictionaries after evaluation, then re-run this cell.

Key metrics to watch:
  - TC Dice: primary success metric (complex SSM targets rotational dynamics)
  - Mean Dice: should be >= V4.6 baseline (~88%)
  - ET Dice: should not regress


## Resume Training (After Disconnect)

> **Important:** Only resume from V5.0 checkpoints. V4.x checkpoints are incompatible.

In [10]:
import os, shutil
os.chdir(REPO_DIR)
os.environ["DRIVE_CKPT_DIR"] = DRIVE_CKPT

resume_ckpt = os.path.join(DRIVE_CKPT, "last.pth")
if os.path.exists(resume_ckpt):
    print(f"Resuming from {resume_ckpt}")
    !python train.py \
        --config configs/textbrats_a100_v5.yaml \
        --resume "{resume_ckpt}" \
        --no-text-ratio 0.15 \
        --grad-accum 2

    sync_checkpoints_to_drive()

    best_ckpt = os.path.join(DRIVE_CKPT, "best_v5.0.pth")
    local_best = os.path.join(REPO_DIR, "checkpoints/best.pth")
    if os.path.exists(local_best):
        shutil.copy2(local_best, best_ckpt)
        print(f"Best checkpoint saved: {best_ckpt}")
else:
    print("No checkpoint to resume from.")
    print(f"Expected: {resume_ckpt}")
    print("Run training first")

Resuming from /content/drive/MyDrive/TextMamba3D/checkpoints/last.pth
2026-03-19 17:07:08.541179: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773940028.564474  123939 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773940028.572322  123939 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773940028.594482  123939 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773940028.594515  123939 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:17739